In [1]:
import dpctl
import dpnp as np
import numba
import numba_dpex as dpex

dpctl.select_default_device().print_device_info()
dpctl.get_devices()

    Name            Intel(R) Xeon(R) CPU           X5680  @ 3.33GHz
    Driver version  2025.19.4.0.18_160000.xmain-hotfix
    Vendor          Intel(R) Corporation
    Filter string   opencl:cpu:0



[<dpctl.SyclDevice [backend_type.opencl, device_type.cpu,  Intel(R) Xeon(R) CPU           X5680  @ 3.33GHz] at 0x7faac3bb8830>]

In [ ]:
import unhash.hash as hash

enc = hash.encode("hello world")
h = hash.hashify(enc)
enc, h

In [ ]:
import dpnp as np
import numba_dpex as dpex


@dpex.dpjit
def f2(a: np.dpnp_array, b: np.dpnp_array) -> np.dpnp_array:
    return 2 * a + 3 * b


N = 2**20

a = np.arange(N, dtype=np.dtype("u4"))
b = np.arange(N, dtype=np.dtype("u4"))
f2(a, b)

In [ ]:
%timeit f2(a,b)

In [ ]:
a = np.array([1, 2, 3, 4], dtype=np.dtype("i4"))
contribution(a)

In [ ]:
%timeit contribution(enc)

In [ ]:
import dpnp as np
import numba_dpex as dpex


@dpex.kernel
def kernel_vector_sum(item, a: np.dpnp_array, b: np.dpnp_array, c: np.dpnp_array):
    i = item.get_id(0)
    c[i] = 2.1 * a[i] + 3.2 * b[i]


N = 20480000

out = np.zeros(N, dtype=np.float64)
a = np.arange(N, dtype=np.float64)
b = np.arange(N, dtype=np.float64)
dpex.call_kernel(kernel_vector_sum, dpex.Range(N), a, b, out)

In [ ]:
%timeit dpex.call_kernel(kernel_vector_sum, dpex.Range(N), a, b, out)

In [ ]:
import numpy as np
from numba import uint32, vectorize


@vectorize([uint32(uint32, uint32)])
def f(a, b):
    return 2 * a + 3 * b


N = 2**20

a = np.arange(N, dtype=np.dtype("u4"))
b = np.arange(N, dtype=np.dtype("u4"))
result = f(a, b)
result

In [ ]:
%timeit f(a,b)

In [42]:
import numpy as np
import numba as numba

cont = np.dtype("u4")

M = cont.type(100000)
Q = cont.type(20000)
MA = M * Q

P0 = np.array([11, 17, 7, 5], cont).reshape(-1, 1)
P1 = np.array([29, 31, 17, 13], cont).reshape(-1, 1)
P2 = np.array([53, 67, 103, 47], cont).reshape(-1, 1)
P3 = np.array([52, 12, 24, 30], cont).reshape(-1, 1)
P4 = np.array([0, 90, 0, 90], cont).reshape(-1, 1)
idx = np.arange(11) + 1

A = np.outer(P0, idx) % P1 + P2
B = A * P3 + P4

@numba.jit(fastmath=True)
def contribution(enc):
    angle = A * enc + B
    sin = np.sin(np.radians(angle))
    return np.round(np.fmod(1 + sin, 0.2) * MA * 5).astype(cont)


@numba.jit
def hashify(enc):
    return np.floor_divide(np.sum(contribution(enc), axis=1) % MA, Q).astype(cont)


enc = np.array([7, 4, 11, 11, 14, 36, 22, 14, 17, 11, 3], cont)

contribution(enc), hashify(enc)

(array([[ 697564737, 1339745962,  435655350,  122147477, 1045284633,
          700807358,  489434837,  122147477,  588190451, 1510565163,
          339555569],
        [ 864545424, 1660444431,  664195735,  849619251,          0,
          691306064,   79116908, 1243626442,  954715367,  697564737,
         1000000000],
        [1736481777, 1877852523, 1735764364, 1000000000,  608268990,
                  0,  694715628, 1877852523, 1735764364, 1659258263,
         1877852523],
        [ 408070965,  568551745, 1253802929, 1877852523,  419218956,
          122147477,  480480962,  691306064,  746197071,  119892464,
         1339745962]], dtype=uint32),
 array([69554, 35256, 40190,  1363], dtype=uint32))

In [43]:
%timeit contribution(enc)
%timeit hashify(enc)
# 5.23 μs ± 196 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)
# 35.9 μs ± 1.41 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)

3.59 μs ± 125 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)
4.43 μs ± 193 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


In [ ]:
import dpnp as np
import numba_dpex as dpex


def contribution(enc: np.dpnp_array):
    P0 = np.array([11, 17, 7, 5], cont).reshape(-1, 1)
    P1 = np.array([29, 31, 17, 13], cont).reshape(-1, 1)
    P2 = np.array([53, 67, 103, 47], cont).reshape(-1, 1)
    P3 = np.array([52, 12, 24, 30], cont).reshape(-1, 1)
    P4 = np.array([0, 90, 0, 90], cont).reshape(-1, 1)

    idx = np.arange(len(enc)).reshape(1, -1)
    angle = (P0 @ (idx + 1) % P1 + P2) * (enc + P3) + P4
    sin = np.sin(np.radians(angle))
    return np.round(np.fmod(1 + sin, 0.2) * MA * 5, dtype=cont)


def hashify(enc: np.dpnp_array):
    return np.floor_divide(np.sum(contribution(enc), axis=1) % MA, Q, dtype=cont)

enc = np.array([7, 4, 11, 11, 14, 36, 22, 14, 17, 11, 3], cont)

contribution(enc), hashify(enc)

(array([[ 697564737, 1339745962,  435655349,  122147477, 1045284632,
          700807357,  489434837,  122147477,  588190451, 1510565162,
          339555568],
        [ 864545423, 1660444431,  664195735,  849619250,          0,
          691306063,   79116908, 1243626441,  954715367,  697564737,
          999999999],
        [1736481776, 1877852522, 1735764363,  999999999,  608268990,
                  0,  694715627, 1877852522, 1735764363, 1659258262,
         1877852522],
        [ 408070965,  568551745, 1253802928, 1877852522,  419218955,
          122147477,  480480961,  691306063,  746197071,  119892463,
         1339745962]], dtype=uint32),
 array([154806,   5759,  95945, 186614], dtype=uint32))

In [8]:
%timeit contribution(enc)
%timeit hashify(enc)

12.3 ms ± 1.17 ms per loop (mean ± std. dev. of 7 runs, 100 loops each)
17.5 ms ± 3.9 ms per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [ ]:
import dpnp as np
import numba_dpex as dpex

cont = np.dtype("u4")

M = 100000
Q = 20000
MA = M * Q


@dpex.kernel(parallel=True, fastmath=True)
def contribution_kernel(item, P0, P1, P2, P3, P4, enc, out):
    i = item.get_id(0)
    j = item.get_id(1)

    angle = (P0[i] * (j + 1)) % P1[i] + P2[i] * (enc[j] + P3[i]) + P4[i]
    sin_val = np.sin(np.radians(angle))
    out[i, j] = cont.type(round(((1 + sin_val) % 0.2) * MA * 5))


@dpex.kernel
def sum_axis1_kernel(item, contrib, row_sums):
    i = item.get_id(0)
    for j in range(11):
        row_sums[i] += contrib[i, j]
        row_sums[i] %= MA
    row_sums[i] /= Q


P0 = np.array([11, 17, 7, 5], cont)
P1 = np.array([29, 31, 17, 13], cont)
P2 = np.array([53, 67, 103, 47], cont)
P3 = np.array([52, 12, 24, 30], cont)
P4 = np.array([0, 90, 0, 90], cont)

out = np.zeros((4, 11), dtype=np.dtype("u4"))
enc = np.array([7, 4, 11, 11, 14, 36, 22, 14, 17, 11, 3], dtype=np.dtype("u4"))


@dpex.dpjit
def hashify_dpex(contrib):
    sums = np.zeros(contrib.shape[0], dtype=cont)
    dpex.call_kernel(sum_axis1_kernel, dpex.Range(4), contrib, sums)
    return sums


dpex.call_kernel(contribution_kernel, dpex.Range(4, 11), P0, P1, P2, P3, P4, enc, out)
out, hashify_dpex(out)

(array([[ 218523993, 1396926208, 1743700648, 1135454576,  297042737,
          608268990,  744318455,  183728166, 1335804265,  480480962,
          427876097],
        [1339745962, 1961946981,  849619251, 1076282953,  191520443,
         1659258263, 1909830056,  946583705,   97319313, 1616288532,
          756373558],
        [1572123903, 1090169944, 1564344650,  756373558,  928932188,
         1271838546, 1563047560, 1439409710,   38053019, 1218693434,
                  0],
        [1591929035,  218523993, 1706796090, 1053416295, 1736481777,
         1053416295, 1907311285, 1825475936, 1735764364,  568551745,
         1932633569]], dtype=uint32),
 array([28606, 20238, 72149, 66515], dtype=uint32))

In [20]:
%timeit dpex.call_kernel(contribution_kernel, dpex.Range(4, 11), P0, P1, P2, P3, P4, enc, out)
%timeit hashify_dpex(out)

882 μs ± 18.7 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
321 μs ± 33.3 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
